In [5]:
import pandas as pd
import numpy as np
import torch
from pathlib import Path
import os
import matplotlib.pyplot as plt
import seaborn as sns
import json

from cuad_cleaning import clean_text_with_map, build_reverse_map, relocate_answer

In [6]:
def find_project_dir_for_my_mac(data_set_path: str) -> Path:
    """Gets training data path"""
    target_path = Path(os.getcwd()).parent / 'data' / 'cuad' / data_set_path #curr dir, go to parent, cd into dataset of choice
    if not target_path.exists():
        print("Path couldn't load!")
    return target_path
target_path = find_project_dir_for_my_mac('train_separate_questions.json')

In [3]:
def load_cuad_df(path):
    """
    Flatten JSON into a tidy DataFrame.
    """
    with open(path) as f:
        raw = json.load(f)

    df = pd.json_normalize(raw["data"], record_path=["paragraphs"], meta=["title"])
    df = df.reset_index(names="doc_id")
    df = df.explode("qas", ignore_index=True) #why explode, at this point there is one row per paragraph but qas inside that row is a list, cant run explode on a qas list
    #horizontally
    df = pd.concat([df.drop(columns="qas"), pd.json_normalize(df["qas"])], axis=1)

    df["category"] = df["question"].str.extract(r'related to "([^"]+)"') #anything in " ... "  in qestion becomes the category for the row
    df = df.explode("answers", ignore_index=True)
    df = pd.concat(
        [df.drop(columns="answers"),
         pd.json_normalize(df["answers"]).rename(columns={"text": "answer_text"})],
        axis=1,
    )
    return df

df = load_cuad_df(target_path)

In [ ]:
# text-cleaning and offset-relocation logic now lives in cuad_cleaning.py, imported above

In [4]:
# make sure every contract title only ever points to ONE version of the text
# (if a title had two different contexts, our cleaning map for it would just pick one and silently drop the other)

unique_title_context_pairs = len(df[["title", "context"]].drop_duplicates())
unique_titles = df["title"].nunique()
if unique_title_context_pairs != unique_titles:
    raise ValueError("found a title with more than one different context")

title_to_clean_text = {}
title_to_reverse_map = {}
for title, context in df.drop_duplicates("title")[["title", "context"]].values:
    cleaned, clean_to_raw = clean_text_with_map(context)
    title_to_clean_text[title] = cleaned
    title_to_reverse_map[title] = build_reverse_map(len(context), clean_to_raw, len(cleaned))

df["text"] = df["title"].map(title_to_clean_text)


In [5]:
annotations = df.apply(relocate_answer, axis=1, title_to_reverse_map=title_to_reverse_map)  # run our offset relocation logic on every row
df = pd.concat([df, annotations], axis=1)

# this is the actual clean df
clean_df = df.rename(columns={"title": "contract_id"})[
    ["contract_id", "id", "text", "category", "is_impossible", "annotation_text", "annotation_start"]
].reset_index(drop=True)

In [ ]:
# double-check nothing got silently mangled while cleaning, separately re-clean each raw answer
# on its own and make sure it still matches what we ended up with. if a row ever doesn't match,
# it means the cleanup accidentally ate part of a real clause instead of just boilerplate.
has_answer = ~clean_df["is_impossible"]

independent_clean = df["answer_text"].apply(lambda s: clean_text_with_map(s)[0] if isinstance(s, str) else None)
mismatch_mask = has_answer & (clean_df["annotation_text"].fillna("") != independent_clean.fillna(""))

flagged = clean_df[mismatch_mask]
print(f"total flagged: {len(flagged)} / {has_answer.sum()} answered rows")
if len(flagged) > 0:
    print(flagged[["contract_id", "category", "annotation_text"]].head(10))